# Background Knowledge in Causal Discovery

Causal discovery algorithms learn graph structure from data alone, but they often
leave some edges undirected or miss orientations. **Background knowledge** lets you
encode domain expertise to constrain the search and resolve ambiguities.

tetrad-port supports three types of knowledge:

1. **Temporal tiers** — variables in earlier tiers cannot be caused by variables in later tiers
2. **Forbidden edges** — explicitly forbid specific directed edges
3. **Required edges** — explicitly require specific directed edges

Knowledge can be specified as:
- A `Knowledge` C++ object (fine-grained control)
- A plain Python `dict` (convenient, auto-converted)

This tutorial uses an 8-variable clinical dataset to demonstrate how each type of
knowledge progressively improves the discovered graph.

In [ ]:
import numpy as np
import pandas as pd
from tetrad_port import TetradPort, Knowledge

tp = TetradPort()

## The Clinical Dataset

We simulate 8 observed variables with this true causal structure:

```
Tier 0 (background):   Age ──────────────────┐
                        Genetics ─────────┐   │
                                          │   │
Tier 1 (lifestyle):    Exercise ◄── Age   │   │
                        Diet              │   │
                        Smoking ──────┐   │   │
                                      │   │   │
Tier 2 (biomarkers):   BMI ◄── Age, Exercise, Diet
                        Cholesterol ◄── Genetics, Diet, Smoking
                        BP ◄── Genetics, BMI, Smoking
                                      │
Tier 3 (outcome):      HeartRisk ◄── Cholesterol, BP
```

The natural temporal ordering (background → lifestyle → biomarkers → outcome)
provides strong prior knowledge that algorithms can exploit.

In [ ]:
np.random.seed(42)
n = 5000

# Tier 0: Background factors
Age = np.random.randn(n)
Genetics = np.random.randn(n)

# Tier 1: Lifestyle factors
Exercise = 0.4 * Age + 0.7 * np.random.randn(n)
Diet = np.random.randn(n)
Smoking = np.random.randn(n)

# Tier 2: Biomarkers
BMI = 0.3 * Age + 0.4 * Exercise + 0.3 * Diet + 0.5 * np.random.randn(n)
Cholesterol = 0.5 * Genetics + 0.4 * Diet + 0.3 * Smoking + 0.5 * np.random.randn(n)
BP = 0.4 * Genetics + 0.3 * BMI + 0.4 * Smoking + 0.5 * np.random.randn(n)

# Tier 3: Outcome
HeartRisk = 0.5 * Cholesterol + 0.5 * BP + 0.4 * np.random.randn(n)

df = pd.DataFrame({
    "Age": Age, "Genetics": Genetics, "Exercise": Exercise,
    "Diet": Diet, "Smoking": Smoking, "BMI": BMI,
    "Cholesterol": Cholesterol, "BP": BP, "HeartRisk": HeartRisk,
})

print(f"Data: {df.shape[0]} samples, {df.shape[1]} variables")
df.head()

## 1. Baseline — No Knowledge

First, run PC with no background knowledge. The algorithm discovers the skeleton
correctly but may leave some edges undirected or miss orientations because the
data alone cannot always distinguish causal direction.

In [ ]:
r_none, g_none = tp.run_pc(df, alpha=0.01)

print(f"Edges found: {r_none['num_edges']}")
print()
for e in sorted(r_none["edges"]):
    marker = "  <<<" if "---" in e else ""
    print(f"  {e}{marker}")
print()
print("<<< = undirected (algorithm could not determine direction)")

## 2. Temporal Tiers

We know the temporal ordering of these variables:
- **Tier 0**: Age, Genetics (fixed at birth)
- **Tier 1**: Exercise, Diet, Smoking (lifestyle choices)
- **Tier 2**: BMI, Cholesterol, BP (biomarkers, measured later)
- **Tier 3**: HeartRisk (outcome)

Temporal tiers forbid edges from later tiers to earlier tiers — BMI cannot cause
Age, and HeartRisk cannot cause Exercise. This resolves ambiguous orientations.

In [ ]:
k_tiers = Knowledge()
k_tiers.set_tier(0, ["Age", "Genetics"])
k_tiers.set_tier(1, ["Exercise", "Diet", "Smoking"])
k_tiers.set_tier(2, ["BMI", "Cholesterol", "BP"])
k_tiers.set_tier(3, ["HeartRisk"])

r_tiers, g_tiers = tp.run_pc(df, alpha=0.01, knowledge=k_tiers)

print(f"Edges found: {r_tiers['num_edges']}")
print()
for e in sorted(r_tiers["edges"]):
    print(f"  {e}")

### What changed?

Compare the baseline (no knowledge) with the tier-constrained result:

In [ ]:
edges_none = set(r_none["edges"])
edges_tiers = set(r_tiers["edges"])

# Find edges that changed orientation or were added/removed
only_baseline = edges_none - edges_tiers
only_tiers = edges_tiers - edges_none

if only_baseline or only_tiers:
    print("Edges in baseline but NOT with tiers:")
    for e in sorted(only_baseline):
        print(f"  - {e}")
    print()
    print("Edges with tiers but NOT in baseline:")
    for e in sorted(only_tiers):
        print(f"  + {e}")
else:
    print("No differences (tiers had no effect on this data)")

### Alternative: Dict-Based Knowledge

Instead of building a `Knowledge` object manually, you can pass a plain dict.
This is equivalent to the Knowledge object above:

In [ ]:
# Dict-based knowledge — automatically converted to Knowledge object
k_dict = {
    "addtemporal": {
        0: ["Age", "Genetics"],
        1: ["Exercise", "Diet", "Smoking"],
        2: ["BMI", "Cholesterol", "BP"],
        3: ["HeartRisk"],
    }
}

# Pass dict directly — no manual conversion needed
r_dict, g_dict = tp.run(df, algorithm="pc", alpha=0.01, knowledge=k_dict)

# Verify same result as Knowledge object
assert set(r_dict["edges"]) == set(r_tiers["edges"]), "Dict and Knowledge results differ!"
print(f"Dict knowledge: {r_dict['num_edges']} edges (matches Knowledge object)")

## 3. Forbidden Edges

Even within the same tier, we may have domain knowledge that certain direct
causal paths are implausible:

- **Forbid Exercise → Cholesterol**: Exercise affects cholesterol indirectly
  through BMI, not directly
- **Forbid Smoking → BMI**: Smoking doesn't directly cause weight changes

These constraints prevent the algorithm from placing edges that contradict
domain expertise, even if statistical associations exist.

In [ ]:
k_forbid = Knowledge()
# Keep the temporal tiers
k_forbid.set_tier(0, ["Age", "Genetics"])
k_forbid.set_tier(1, ["Exercise", "Diet", "Smoking"])
k_forbid.set_tier(2, ["BMI", "Cholesterol", "BP"])
k_forbid.set_tier(3, ["HeartRisk"])

# Add forbidden edges
k_forbid.set_forbidden("Exercise", "Cholesterol")
k_forbid.set_forbidden("Smoking", "BMI")

r_forbid, g_forbid = tp.run_pc(df, alpha=0.01, knowledge=k_forbid)

print(f"Edges found: {r_forbid['num_edges']}")
print()
for e in sorted(r_forbid["edges"]):
    print(f"  {e}")

# Verify forbidden edges are absent
print()
print("Verification:")
print(f"  Exercise -> Cholesterol present? {'Exercise --> Cholesterol' in r_forbid['edges']}")
print(f"  Smoking -> BMI present? {'Smoking --> BMI' in r_forbid['edges']}")

## 4. Required Edges

Sometimes we know from prior studies or established science that certain causal
links exist. Required edges force these into the output:

- **Require Smoking → BP**: Well-established that smoking raises blood pressure
- **Require Diet → Cholesterol**: Dietary fat intake directly affects cholesterol

This is especially useful when sample size is too small for the algorithm to
detect a true edge, or when the edge orientation would otherwise be ambiguous.

In [ ]:
k_full = Knowledge()
# Temporal tiers
k_full.set_tier(0, ["Age", "Genetics"])
k_full.set_tier(1, ["Exercise", "Diet", "Smoking"])
k_full.set_tier(2, ["BMI", "Cholesterol", "BP"])
k_full.set_tier(3, ["HeartRisk"])

# Forbidden edges
k_full.set_forbidden("Exercise", "Cholesterol")
k_full.set_forbidden("Smoking", "BMI")

# Required edges
k_full.set_required("Smoking", "BP")
k_full.set_required("Diet", "Cholesterol")

r_full, g_full = tp.run_pc(df, alpha=0.01, knowledge=k_full)

print(f"Edges found: {r_full['num_edges']}")
print()
for e in sorted(r_full["edges"]):
    print(f"  {e}")

# Verify required edges are present
print()
print("Verification:")
print(f"  Smoking -> BP present? {'Smoking --> BP' in r_full['edges']}")
print(f"  Diet -> Cholesterol present? {'Diet --> Cholesterol' in r_full['edges']}")

## Summary — Progressive Refinement

Compare all four configurations side by side:

In [ ]:
configs = [
    ("No knowledge", r_none),
    ("+ Temporal tiers", r_tiers),
    ("+ Forbidden edges", r_forbid),
    ("+ Required edges", r_full),
]

# Count directed vs undirected
for name, r in configs:
    directed = sum(1 for e in r["edges"] if "-->" in e)
    undirected = sum(1 for e in r["edges"] if "---" in e)
    print(f"{name:25s}  edges={r['num_edges']:2d}  directed={directed:2d}  undirected={undirected:2d}")

In [ ]:
# Compare final result against the true causal graph
true_edges = {
    ("Age", "Exercise"), ("Age", "BMI"),
    ("Exercise", "BMI"), ("Diet", "BMI"), ("Diet", "Cholesterol"),
    ("Genetics", "Cholesterol"), ("Genetics", "BP"),
    ("Smoking", "Cholesterol"), ("Smoking", "BP"),
    ("BMI", "BP"),
    ("Cholesterol", "HeartRisk"), ("BP", "HeartRisk"),
}

discovered_directed = set(g_full["directed_edges"])

correct = true_edges & discovered_directed
missed = true_edges - discovered_directed
extra = discovered_directed - true_edges

print(f"True edges: {len(true_edges)}")
print(f"Correctly discovered (directed): {len(correct)} / {len(true_edges)}")
print()
if missed:
    print("Missed:")
    for e in sorted(missed):
        print(f"  {e[0]} --> {e[1]}")
if extra:
    print("Extra (false positives):")
    for e in sorted(extra):
        print(f"  {e[0]} --> {e[1]}")
if not missed and not extra:
    print("Perfect recovery!")

## Bonus: Forbid Edges Within a Tier

Sometimes variables in the same tier are known to be independent of each other.
For example, Age and Genetics (both tier 0) have no causal relationship between
them. `set_tier_forbidden_within()` forbids all directed edges between variables
in the same tier.

In [ ]:
k_within = Knowledge()
k_within.set_tier(0, ["Age", "Genetics"])
k_within.set_tier(1, ["Exercise", "Diet", "Smoking"])
k_within.set_tier(2, ["BMI", "Cholesterol", "BP"])
k_within.set_tier(3, ["HeartRisk"])

# Forbid edges within tier 0 (Age <-> Genetics) and tier 1
k_within.set_tier_forbidden_within(0, True)
k_within.set_tier_forbidden_within(1, True)

r_within, g_within = tp.run_pc(df, alpha=0.01, knowledge=k_within)

print(f"Edges found: {r_within['num_edges']}")
print()
for e in sorted(r_within["edges"]):
    print(f"  {e}")

print()
print("Note: No edges between Age-Genetics or Exercise-Diet-Smoking")